# Thực hành xây dựng mô hình ngôn ngữ N-Gram

Notebook này hướng dẫn từng bước:

1. Đọc và tiền xử lý corpus.
2. Xây dựng Unigram, Bigram và Trigram.
3. Tính xác suất từ tiếp theo và xác suất câu.
4. Đánh giá bằng perplexity.
5. Sinh văn bản.

Mã nguồn mô hình nằm trong file `ngram_language_model.py` cùng thư mục.

## 1. Import thư viện và mô hình

Chương trình chỉ dùng thư viện chuẩn Python, không cần cài thêm package.

In [1]:
from pathlib import Path

from ngram_language_model import (
    NGramLanguageModel,
    tach_cau,
    tach_tu,
)

print('Đã import mô hình thành công.')

Đã import mô hình thành công.


## 2. Đọc corpus

Corpus mẫu gồm các câu ngắn liên quan đến xử lý ngôn ngữ tự nhiên. Khi áp dụng thực tế, có thể thay bằng file văn bản khác.

In [2]:
duong_dan = Path('du_lieu_mau.txt')
van_ban = duong_dan.read_text(encoding='utf-8')

print('Số ký tự:', len(van_ban))
print('Số câu:', len(tach_cau(van_ban)))
print('\nBa câu đầu:')
for cau in tach_cau(van_ban)[:3]:
    print('-', cau)

Số ký tự: 1485
Số câu: 24

Ba câu đầu:
- Xử lý ngôn ngữ tự nhiên giúp máy tính hiểu và tạo ra ngôn ngữ của con người
- Mô hình ngôn ngữ gán xác suất cho một chuỗi từ
- Mô hình ngôn ngữ cũng dự đoán từ tiếp theo dựa trên ngữ cảnh


## 3. Tách câu và tách token

Bản minh họa tách theo đơn vị chữ để mã nguồn đơn giản. Với dự án tiếng Việt thực tế, nên dùng thêm một bộ tách từ để ghép các từ đa tiếng như “xử lý” hoặc “ngôn ngữ”.

In [3]:
cau_vi_du = 'Mô hình ngôn ngữ dự đoán từ tiếp theo.'
print('Câu gốc:', cau_vi_du)
print('Các token:', tach_tu(cau_vi_du))

Câu gốc: Mô hình ngôn ngữ dự đoán từ tiếp theo.
Các token: ['mô', 'hình', 'ngôn', 'ngữ', 'dự', 'đoán', 'từ', 'tiếp', 'theo']


## 4. Huấn luyện mô hình Bigram

Bigram dùng một token đứng trước làm ngữ cảnh:

$$
P(w_i\mid w_{i-1})
=\frac{C(w_{i-1},w_i)+k}{C(w_{i-1})+k|V|}.
$$

Ở đây dùng add-$k$ smoothing với $k=0{,}1$.

In [4]:
bigram = NGramLanguageModel(n=2, smoothing_k=0.1)
bigram.fit(van_ban)

print('Số câu huấn luyện:', bigram.so_cau_huan_luyen)
print('Kích thước từ vựng:', len(bigram.vocab))
print('Số ngữ cảnh đã gặp:', len(bigram.context_counts))

Số câu huấn luyện: 24
Kích thước từ vựng: 153
Số ngữ cảnh đã gặp: 152


## 5. Quan sát bảng đếm

Với ngữ cảnh `mô`, ta xem các token đã xuất hiện ngay sau nó trong corpus.

In [5]:
ngu_canh = ('mô',)
print('Số lần ngữ cảnh xuất hiện:', bigram.context_counts[ngu_canh])
print('Các token theo sau và số lần xuất hiện:')
print(bigram.ngram_counts[ngu_canh].most_common())

Số lần ngữ cảnh xuất hiện: 11
Các token theo sau và số lần xuất hiện:
[('hình', 11)]


## 6. Dự đoán token tiếp theo

Phương thức `top_tu_tiep_theo()` trả về các token có xác suất cao nhất sau ngữ cảnh đã cho.

In [6]:
for ngu_canh_text in ['mô hình', 'ngôn ngữ', 'dữ liệu']:
    du_doan = bigram.top_tu_tiep_theo(tach_tu(ngu_canh_text), so_luong=5)
    print(f'\nSau {ngu_canh_text!r}:')
    for tu, xac_suat in du_doan:
        print(f'  {tu:<15} {xac_suat:.6f}')


Sau 'mô hình':
  ngôn            0.155894
  học             0.041825
  có              0.041825
  n               0.041825
  bigram          0.041825

Sau 'ngôn ngữ':
  cảnh            0.242321
  tự              0.071672
  của             0.037543
  </s>            0.037543
  gán             0.037543

Sau 'dữ liệu':
  dùng            0.060109
  </s>            0.060109
  huấn            0.060109
  hình            0.005464
  trạng           0.005464


## 7. Tính xác suất câu

Xác suất câu là tích của các xác suất có điều kiện. Chương trình cộng log xác suất để tránh sai số do nhân nhiều số rất nhỏ.

In [7]:
cac_cau = [
    'mô hình ngôn ngữ dự đoán từ tiếp theo',
    'dữ liệu huấn luyện càng lớn',
    'từ tiếp theo mô hình dự đoán',
]

for cau in cac_cau:
    print('\nCâu:', cau)
    print('Log xác suất:', bigram.log_xac_suat_cau(cau))
    print('Xác suất:', f'{bigram.xac_suat_cau(cau):.6e}')


Câu: mô hình ngôn ngữ dự đoán từ tiếp theo
Log xác suất: -22.20471785684226
Xác suất: 2.273074e-10

Câu: dữ liệu huấn luyện càng lớn
Log xác suất: -20.93585756214117
Xác suất: 8.084862e-10

Câu: từ tiếp theo mô hình dự đoán
Log xác suất: -26.249021722007434
Xác suất: 3.982857e-12


## 8. Đánh giá bằng perplexity

$$
PP(W)=\exp\left(-\frac{1}{M}\sum_{i=1}^{M}\log P(w_i\mid h_i)\right).
$$

Perplexity thấp hơn thường biểu thị khả năng dự đoán tốt hơn, nếu các mô hình dùng cùng dữ liệu và cùng cách tách token.

In [8]:
tap_kiem_thu = '''
Mô hình ngôn ngữ dự đoán từ tiếp theo.
Làm trơn giúp xử lý những cụm từ chưa xuất hiện.
'''

print('Perplexity của Bigram:', bigram.perplexity(tap_kiem_thu))

Perplexity của Bigram: 14.782012178170577


## 9. Sinh văn bản

Mô hình bắt đầu từ token đầu câu, lấy mẫu token tiếp theo rồi lặp lại cho tới token kết thúc. Tham số `seed` giúp tái tạo cùng một kết quả.

In [9]:
for i in range(5):
    print(f'{i + 1}.', bigram.sinh_cau(seed=100 + i))

1. mô hình dự đoán các từ một vào số đếm của lĩnh vực
2. skip gram dự đoán tốt hơn
3. mô hình ngôn ngữ cảnh
4. tập kiểm thử phải tách biệt với tập kiểm thử phải tách biệt với tập kiểm thử phải tách biệt
5. tập dữ liệu


## 10. So sánh Unigram, Bigram và Trigram

N lớn hơn cho phép mô hình dùng ngữ cảnh dài hơn, nhưng cũng làm dữ liệu thưa hơn.

In [10]:
ket_qua = []

for n in (1, 2, 3):
    mo_hinh = NGramLanguageModel(n=n, smoothing_k=0.1).fit(van_ban)
    pp = mo_hinh.perplexity(tap_kiem_thu)
    cau_sinh = mo_hinh.sinh_cau(seed=42)
    ket_qua.append((n, len(mo_hinh.context_counts), pp, cau_sinh))

print('{:<5} {:<15} {:<15} {}'.format('N', 'Số ngữ cảnh', 'Perplexity', 'Câu sinh'))
for n, so_ngu_canh, pp, cau_sinh in ket_qua:
    print(f'{n:<5} {so_ngu_canh:<15} {pp:<15.6f} {cau_sinh}')

N     Số ngữ cảnh     Perplexity      Câu sinh
1     1               72.914734       những ngôn gán mô tốt khả transformer nhiên theo ngôn mô các ngôn
2     152             14.782012       cbow dự đoán từ ngữ cảnh dài
3     251             24.010889       cbow dự đoán từ trung tâm dựa trên các từ ngữ cảnh


## 11. Thử nghiệm với corpus của riêng em

Đổi đường dẫn phía dưới thành một file UTF-8 khác. Nên dùng riêng dữ liệu huấn luyện và dữ liệu kiểm thử.

In [11]:
from pathlib import Path
from ngram_language_model import NGramLanguageModel

# Đọc bộ dữ liệu riêng đã chuẩn bị
bo_corpus_rieng = Path('corpus_cua_em.txt').read_text(encoding='utf-8')
mo_hinh_rieng = NGramLanguageModel(n=3, smoothing_k=0.1).fit(bo_corpus_rieng)

print(f'✅ Số câu huấn luyện trong corpus riêng: {mo_hinh_rieng.so_cau_huan_luyen}')
print(f'✅ Kích thước từ vựng: {len(mo_hinh_rieng.vocab)} từ.')
print(f'✅ Số ngữ cảnh Trigram đã học: {len(mo_hinh_rieng.context_counts)}')
print('Câu sinh thử nghiệm 1:', mo_hinh_rieng.sinh_cau(seed=1))
print('Câu sinh thử nghiệm 2:', mo_hinh_rieng.sinh_cau(seed=42))


✅ Số câu huấn luyện trong corpus riêng: 18
✅ Kích thước từ vựng: 190 từ.
✅ Số ngữ cảnh Trigram đã học: 182
Câu sinh thử nghiệm 1: mô hình tuần tự như rnn và lstm giải quyết triệt để vấn đề xác suất của từ tiếp theo
Câu sinh thử nghiệm 2: biểu diễn vector văn bản tự nhiên là một nhánh quan trọng của trí tuệ nhân tạo


## 12. Câu hỏi tự kiểm tra & Trả lời chi tiết

**1. Vì sao xác suất của cả câu có thể bằng 0 khi không dùng smoothing?**  
> *Trả lời:* Theo công thức xác suất dây chuyền: $P(w_1,\dots,w_m) = \prod_{i=1}^m P(w_i\mid w_{i-N+1}^{i-1})$. Khi xuất hiện dù chỉ một N-gram chưa từng thấy trong tập huấn luyện (Unseen N-gram), tần suất của nó bằng 0 làm cho xác suất có điều kiện $P = 0$. Một thừa số bằng 0 sẽ khiến tích của toàn bộ câu bị triệt tiêu thành 0 (hiện tượng Zero Probability). Kỹ thuật Smoothing (Laplace, Add-$k$) giải quyết vấn đề này bằng cách phân bổ lại một lượng xác suất nhỏ cho các trường hợp chưa xuất hiện.

**2. Bigram và Trigram khác nhau ở độ dài ngữ cảnh như thế nào?**  
> *Trả lời:* 
> - **Bigram ($N=2$):** Sử dụng độ dài ngữ cảnh là $N-1 = 1$ từ đứng ngay trước ($w_{i-1}$) để dự đoán từ tiếp theo: $P(w_i\mid w_{i-1})$.
> - **Trigram ($N=3$):** Sử dụng độ dài ngữ cảnh là $N-1 = 2$ từ đứng trước ($w_{i-2}, w_{i-1}$) để dự đoán từ tiếp theo: $P(w_i\mid w_{i-2}, w_{i-1})$. Trigram nắm bắt ngữ nghĩa và cấu trúc cú pháp tốt hơn nhưng đòi hỏi không gian lưu trữ và kích thước tập dữ liệu lớn hơn nhiều để tránh dữ liệu thưa thớt.

**3. Vì sao nên tính log xác suất thay vì nhân trực tiếp nhiều xác suất?**  
> *Trả lời:* 
> - **Tránh tràn số dưới (Floating-point Underflow):** Do các xác suất $P(w_i\mid \dots) \in (0, 1]$, tích của nhiều số nhỏ trong câu dài sẽ tiệm cận 0 rất nhanh và vượt quá giới hạn lưu trữ của số chấm động trên máy tính.
> - **Tối ưu tính toán:** Hàm $\log$ biến phép nhân thành phép cộng: $\log \prod P_i = \sum \log P_i$. Phép cộng trên máy tính vừa bảo toàn độ chính xác số học, vừa có tốc độ xử lý nhanh hơn phép nhân.

**4. Vì sao không nên so sánh perplexity của hai mô hình dùng cách tách token khác nhau?**  
> *Trả lời:* Chỉ số Perplexity phụ thuộc chặt chẽ vào số lượng token $M$ trong văn bản kiểm thử ($PP = \exp(-\frac{1}{M}\sum \log P)$) và độ lớn từ vựng $|V|$. Nếu hai mô hình dùng phương pháp phân tách token khác nhau (ví dụ: tách theo từ đơn tiếng Việt vs tách theo âm tiết/subword), số lượng $M$ và phân bố từ vựng sẽ không đồng nhất, dẫn đến giá trị Perplexity không còn cùng một thang đo chuẩn để so sánh công bằng.

**5. Điều gì xảy ra với số lượng N-Gram khi tăng $N$?**  
> *Trả lời:* Về mặt lý thuyết, số lượng N-gram tối đa tăng theo hàm số mũ $|V|^N$. Trong thực tế, khi $N$ tăng lên, số lượng N-gram độc nhất tăng vọt, trong khi đại đa số N-gram chỉ xuất hiện 1 lần (Hapax Legomena). Điều này dẫn đến hiện tượng dữ liệu cực kỳ thưa thớt (Data Sparsity), đòi hỏi lượng dữ liệu khổng lồ và các giải thuật làm trơn tinh vi hơn (Backoff, Kneser-Ney).

---

## 13. Tổng kết Thực nghiệm Mô hình Ngôn ngữ N-Gram (Summary)

### Data Analysis Key Findings
* **Unigram ($N=1$):** Đạt Perplexity **72.91** (cao nhất), câu sinh mang tính liệt kê từ tần suất cao, không có sự liên kết ngữ pháp.
* **Bigram ($N=2$):** Đạt Perplexity **14.78** (thấp nhất và tối ưu trên tập dữ liệu mẫu), tạo ra câu có tính gắn kết ngữ nghĩa hợp lý giữa 2 từ kế tiếp.
* **Trigram ($N=3$):** Đạt Perplexity **24.01**, số lượng ngữ cảnh tăng từ 152 lên 251. Trigram sinh câu rất mượt mà theo cấu trúc chuyên ngành nhưng chịu ảnh hưởng của độ thưa dữ liệu do tập mẫu nhỏ.

### Insights or Next Steps
* Với tập dữ liệu kích thước nhỏ hoặc trung bình, mô hình Bigram kết hợp Add-$k$ Smoothing ($k=0.1$) đem lại sự cân bằng tốt nhất giữa độ phức tạp tính toán và độ chính xác dự đoán.
* Để nâng cao chất lượng mô hình N-gram trên tiếng Việt, cần tích hợp bộ tách từ chuyên biệt (Word Segmentation) và mở rộng kích thước corpus huấn luyện đa lĩnh vực.
